In [ ]:
import time
from PIL import Image
import numpy as np
import scipy.cluster
import serial
import kociemba

# ArduinoCommunicator class definition
class ArduinoCommunicator:
    def __init__(self, port="/dev/ttyUSB0", baudrate=9600, timeout=2):
        self.port = port
        self.baudrate = baudrate
        self.timeout = timeout
        self.ser = None
        self.connected = False

    def connect(self):
        try:
            self.ser = serial.Serial(self.port, self.baudrate, timeout=self.timeout)
            time.sleep(2) # Aguarda a inicialização da comunicação serial
            self.connected = True
            print(f"[ArduinoCommunicator] Conectado à porta serial: {self.port}")
            return True
        except serial.SerialException as e:
            print(f"[ArduinoCommunicator] ERRO CRÍTICO: Não foi possível abrir a porta serial {self.port}: {e}")
            print("[ArduinoCommunicator] Certifique-se de que o dispositivo está conectado e que você tem permissão.")
            self.connected = False
            return False

    def disconnect(self):
        if self.ser and self.ser.is_open:
            self.ser.close()
            self.connected = False
            print("[ArduinoCommunicator] Conexão serial encerrada.")

    def send_command(self, cmd):
        if not self.connected:
            print(f"[ArduinoCommunicator] Erro: Não conectado ao Arduino. Comando '{cmd}' não enviado.")
            return False
        try:
            self.ser.write((cmd + "\n").encode("utf-8"))
            # print(f"[ArduinoCommunicator] Enviado: {cmd}") # Descomente para debug
            return True
        except serial.SerialException as e:
            print(f"[ArduinoCommunicator] Erro ao enviar comando '{cmd}': {e}")
            self.connected = False
            return False

    def read_line(self):
        if not self.connected:
            # print("[ArduinoCommunicator] Não conectado ao Arduino. Não foi possível ler.") # Descomente para debug
            return None
        try:
            line = self.ser.readline().decode("utf-8", errors="ignore").strip()
            # if line: print(f"[ArduinoCommunicator] Recebido: {line}") # Descomente para debug
            return line
        except serial.SerialException as e:
            print(f"[ArduinoCommunicator] Erro ao ler da serial: {e}")
            self.connected = False
            return None

PORT = "/dev/ttyUSB0"   # ajuste para COM3, COM4 etc. no Windows
BAUD = 9600

SCAN_ORDER = ['U', 'L', 'F', 'R', 'B', 'D']
REQUIRED_ORDER = ['U', 'R', 'F', 'D', 'L', 'B']


def analyse(image_path):
    if not image_path:
        raise ValueError("image_path é obrigatório.")

    img = Image.open(image_path)
    img = img.rotate(316)
    img = img.crop((310, 445, 675, 810))

    width, _ = img.size
    incr = width / 3.0

    colours = []
    for i in range(3):
        for j in range(3):
            im = img.crop((incr * j, incr * i, incr * (j + 1), incr * (i + 1))).resize((50, 50))
            ar = np.asarray(im)
            shape = ar.shape
            ar = ar.reshape(np.prod(shape[:2]), shape[2]).astype(float)

            num_clusters = 6
            codes, _ = scipy.cluster.vq.kmeans(ar, num_clusters)
            vecs, _ = scipy.cluster.vq.vq(ar, codes)
            counts, _ = np.histogram(vecs, len(codes))
            index_max = np.argmax(counts)
            peak = codes[index_max]
            colours.append([peak[0], peak[1], peak[2]])

    hsl = []
    actual = []

    for colour in colours:
        r = colour[0] / 255.0
        g = colour[1] / 255.0
        b = colour[2] / 255.0
        mx = max(r, g, b)
        mn = min(r, g, b)
        l = (mn + mx) / 2.0

        if mx == mn:
            h = 0
            s = 0
        else:
            if l < 0.5:
                s = (mx - mn) / (mx + mn)
            else:
                s = (mx - mn) / (2.0 - mx - mn)

            if mx == r:
                h = (g - b) / (mx - mn)
            elif mx == g:
                h = 2.0 + (b - r) / (mx - mn)
            else:
                h = 4.0 + (r - g) / (mx - mn)

            h *= 60
            if h < 0:
                h += 360

        hsl.append((h, s, l))
        actual.append("w" if s < 0.45 else "unknown")

    known_colours = {
        "r": (342, 0.80),
        "b": (194, 0.98),
        "g": (128, 1.00),
        "y": (78, 1.00),
        "o": (13.7, 0.72),
    }

    for i in range(len(hsl)):
        if actual[i] != "w":
            best_dist = float("inf")
            best_colour = "unknown"
            h, s, _ = hsl[i]

            for key, (kh, ks) in known_colours.items():
                dist = abs(h - kh) + abs((s * 360) - (ks * 360))
                if dist < best_dist:
                    best_dist = dist
                    best_colour = key

            actual[i] = best_colour

    result = ''.join(actual)
    print(f"Detected: {result}")
    return result


def cube_string_notation(colours, required_order):
    mapping = {}
    for idx, face in enumerate(required_order):
        mapping[colours[idx * 9 + 4]] = face

    for color, face in mapping.items():
        colours = colours.replace(color, face)

    return colours


def expand_solution(solution):
    moves = []
    for mov in solution.split():
        if mov.endswith("2"):
            moves.extend([mov[0], mov[0]])
        elif mov.endswith("'"):
            moves.append(mov[0].lower())
        else:
            moves.append(mov[0].upper())
    return moves


def main():
    cube = {}
    scan_order = SCAN_ORDER.copy()

    comm = ArduinoCommunicator(PORT, BAUD)
    if not comm.connect():
        print("Falha ao conectar ao Arduino. Encerrando.")
        return

    try:
        print("PC pronto. Aguardando Arduino...")

        while True:
            line = comm.read_line()
            if not line:
                continue

            print("RX:", line)

            if line == "READY":
                comm.send_command("PC_READY")

            elif line == "SCAN":
                if not scan_order:
                    comm.send_command("RESET")
                    scan_order = SCAN_ORDER.copy()
                    cube.clear()
                    continue

                face = scan_order.pop(0)
                cube[face] = analyse("current_face.png")
                comm.send_command("NEXT")

            elif line == "SOLVE":
                if any(face not in cube for face in REQUIRED_ORDER):
                    comm.send_command("ERROR")
                    continue

                cube_formatted = ''.join(cube[face] for face in REQUIRED_ORDER)
                cube_notation = cube_string_notation(cube_formatted, REQUIRED_ORDER)
                solution = kociemba.solve(cube_notation)
                print("Solução:", solution)

                moves = expand_solution(solution)
                for mv in moves:
                    comm.send_command(f"MOVE {mv}")
                    ack = comm.read_line()
                    if ack != "DONE":
                        print("Aviso: resposta inesperada:", ack)

                comm.send_command("FINISH")
                print("Processo finalizado.")
                break

            elif line == "RESET":
                cube.clear()
                scan_order = SCAN_ORDER.copy()

            elif line == "ERROR":
                print("Erro recebido do Arduino.")
    finally:
        comm.disconnect()


if __name__ == "__main__":
    main()
